## RAG Pipeline:- Data Ingestion to VectorDB

In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
## Read all pdf files in the directory

def process_all_pdfs(pdf_directory):
    '''ALL pdf files will be processed'''

    all_documents = []
    pdf_dir=Path(pdf_directory)

    ## Find all pdf files
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    print(f'Found {len(pdf_files)} PDF files to process')

    for pdf_file in pdf_files:
        print(f'\nProcessing: {pdf_file.name}')
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            ## Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f'Loaded {len(documents)} pages')

        except Exception as e:
            print(f'Error: {e}')

    print(f'\nTotal documents loaded; {len(all_documents)}')
    return all_documents

## Process all PDFs in data directory
all_pdf_documents = process_all_pdfs('../data')

    
    

Found 4 PDF files to process

Processing: Resume_Meghana.pdf
Loaded 1 pages

Processing: shyam resume.pdf
Loaded 3 pages

Processing: Kaushik_ML_Resume.pdf
Loaded 1 pages

Processing: Kaushik_ML_DEV_Resume.pdf
Loaded 1 pages

Total documents loaded; 6


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}, page_content='Meghana Perada                                                                                                 +91-8374955043 \nRoll No.:23011M2102                                                                                                       \nmeghanaperada9@gmail.com \nBachelor of Technology                                                                             linkedin.com/in/meghana-perada-035639348 \nJawaharlal Nehru Technological Un

### Chunking using text_splitter(RecursiveCharacterTextSplitter)

In [ ]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,  ##size of chunk
        chunk_overlap=chunk_overlap,    ##overlap 200 character from previous chunk into present chunk
        length_function=len,    ##use python len() function to count no.of characters
        separators=["\n\n", "\n", " ", ""]  ##seperate chunks based on para, lines, space, character in priority wise
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 6 documents into 18 chunks

Example chunk:
Content: Meghana Perada                                                                                                 +91-8374955043 
Roll No.:23011M2102                                                      ...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}, page_content='Meghana Perada                                                                                                 +91-8374955043 \nRoll No.:23011M2102                                                                                                       \nmeghanaperada9@gmail.com \nBachelor of Technology                                                                             linkedin.com/in/meghana-perada-035639348 \nJawaharlal Nehru Technological Un

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/kaushik1707/Desktop/Trust-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8970.56it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/qv/_yq6cw0d01bgq375mdy816400000gn/T/ipykernel_12445/540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [4]:
text = ["RAG helps LLMs retrieve relevant information from documents."]

embedding = embedding_manager.generate_embeddings(text)

print(embedding)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]

Generated embeddings with shape: (1, 384)
[[-3.69041003e-02  8.92806128e-02  2.98138033e-03 -2.62994021e-02
   3.13861598e-03  2.30778325e-02 -3.20943110e-02  3.56700011e-02
   1.74927600e-02  3.46220960e-03 -3.30814184e-03  9.27609503e-02
   5.76529130e-02 -6.11839518e-02 -3.16206440e-02  8.21428373e-02
   7.09013194e-02  1.12859674e-01 -2.17528958e-02 -3.73659283e-02
  -2.97641791e-02  4.72569764e-02  3.68978195e-02 -3.25906537e-02
   1.88353844e-02  4.19474207e-02 -7.12235793e-02  9.28270631e-03
   4.61463705e-02 -5.84782138e-02  3.48752812e-02  9.00129676e-02
  -2.93497555e-02  3.89232561e-02 -4.85700183e-03  2.98769139e-02
  -1.09920732e-03  9.83001292e-02 -1.15468130e-02 -1.32145118e-02
   5.08413138e-03 -1.90128908e-02 -7.14813313e-03 -1.86184291e-02
   3.43922526e-02  3.60177979e-02 -1.17584337e-02  2.98321936e-02
  -5.56017831e-02  3.38944532e-02 -5.47634475e-02 -2.80610751e-02
   1.91606884e-03  4.35059927e-02 -4.84762974e-02 -2.96514053e-02
   6.52071163e-02 -2.13471092e-02 